# 14 — Alberta director outputs: the curated few for the presentation (mirror of the Y2Y-wide `21_director_outputs`)

Every output here is ONE call to a shared function in `director_plot.py`; `13_figures` (the complete record) renders its copy
of the same asset with the same call, so a change in the module — or a knob changed in `dp.STYLE` below — reaches both.
The package is loaded through `dp.load(...)` with the Alberta grid, manifest, the Upper Smoky areas of interest in the role the
declared IPCA proposals play on the Y2Y-wide maps, and the Alberta window; **Act 1 is built as on the Y2Y-wide deck — the frame at
left and ONE zoom inset on the key core cluster (the largest regional cluster by default; `dp.STYLE["inset_clusters"]`).**
Outputs → `director_package/director_outputs/`. Kernel `y2y-geo`; ~2 min.

In [ ]:
import importlib, json, pathlib, sys, textwrap
from types import SimpleNamespace
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc, director_plot as dp
for _m in (config, lc, ec, dc, dp):
    importlib.reload(_m)
HERE = ROOT / "analyses" / "alberta_prioritization"; PKG = HERE / "director_package"; GEO, TAB = PKG / "geotiffs", PKG / "tables"
assert (PKG / "summary.json").exists(), "run 12_tiers_and_clusters first"
S = json.loads((PKG / "summary.json").read_text())
assert S.get("version", "v1") == config.Y2Y_VERSION, f"the package on disk is {S.get('version', 'v1')}, config is {config.Y2Y_VERSION} -- run 12 first"
# the Alberta grid as a director_core-compatible object (the parent grid masked to Alberta: same shape, same transform)
AB = config.AB_HANDOFF_DIR
with rasterio.open(AB / "cost_uniform.tif") as src:
    tr, shp, prof = src.transform, src.shape, src.profile
pu = lc.pu_mask(AB)
with rasterio.open(AB / "mask_protected_areas.tif") as src:
    locked2d = (src.read(1) == 1) & pu
G = SimpleNamespace(pu=pu, locked2d=locked2d, locked=locked2d[pu], disc=~locked2d[pu], n_pu=int(pu.sum()), n_disc=int((~locked2d[pu]).sum()),
                    shape=shp, transform=tr, crs=config.TARGET_CRS, profile=prof, cell_km2=abs(tr.a * tr.e) / 1e6)
G.rows, G.cols = np.where(pu)
rows_, cols_ = np.where(G.pu.any(axis=1))[0], np.where(G.pu.any(axis=0))[0]
R0, R1, C0, C1 = max(rows_.min() - 10, 0), min(rows_.max() + 11, G.shape[0]), max(cols_.min() - 10, 0), min(cols_.max() + 11, G.shape[1])
WIN = (float(C0), float(C1), float(R0), float(R1))                                       # the Alberta strip (x0, x1, y_top, y_bottom)
AOI = gpd.read_file(GEO / "aoi.gpkg")                                                    # the Upper Smoky Nature-First zone + SRP planning area (12)
OVERLAY = SimpleNamespace(gdf=AOI.assign(ls=["-" if "Nature-First" in n else (0, (4, 2)) for n in AOI.name]), mask2d=None,
                          label="Upper Smoky areas of interest (not locked in)")
# presentation knobs -- ONE place; edit and re-run (the Y2Y-wide values stay in director_plot.STYLE; these are the Alberta settings)
AB_TOWNS = tuple(t for t in ("Jasper", "Edmonton", "Calgary", "Banff") if t in dc.Y2Y_TOWNS)
dp.STYLE.update(pa_layer_min_km2=50,            # named-PA floor for the insets / locators (read at load)
                window_scale_km=100,            # the scale bar on the Alberta frame
                inset_min_km=160, inset_pad_km=25,   # the zoom window around the key cluster (Y2Y-wide: 320 / 45)
                towns=AB_TOWNS, wide_main_towns=AB_TOWNS, wide_main_names="abbrev", lat53=False,
                map_layout="wide")              # slide-shaped: the frame at left, the inset at right
C = dp.load(pkg=PKG, grid=G, manifest=config.ab_paths().manifest, overlay=OVERLAY, window=WIN, hex_grid=False)
globals().update({k: v for k, v in vars(C).items() if not k.startswith("_")})
OUT = PKG / "director_outputs"; OUT.mkdir(exist_ok=True)
KEY = int(PICKS[PICKS.act == "Act 1"].sort_values("km2", ascending=False).iloc[0].number) if len(PICKS[PICKS.act == "Act 1"]) else None
dp.STYLE["inset_clusters"] = (KEY,) if KEY is not None else ()            # ONE inset on the key core cluster -- override: dp.STYLE["inset_clusters"] = (2,)
plt.rcParams.update(dp.SPEC_RC)          # every figure takes the table spec's type (Cronos Pro, ink/cap/mut)
dp.STYLE["titles"] = False               # the slide carries the title (as the Y2Y-wide 21); 13 keeps titles
print(f"package {S['version']} | {len(S['forms'])} positions | band {S['band']} | key cluster {KEY} | outputs -> {OUT.relative_to(ROOT)}")


In [ ]:
# ---- 1. the values in the analysis (the Alberta rows from 12's record, rendered to the table spec) ----------------------------------
ROWS0 = pd.read_csv(TAB / "T-D0_values_rows.csv").fillna("")
dp.values_table(C, OUT / "01_values_table.png", "Y2Y Alberta Objectives Hierarchy", rows=ROWS0[dp.VALUES_COLUMNS].values.tolist(), metrics=ROWS0.metric.tolist(),
                label="Y2Y SPATIAL DECISION TOOL  ·  ALBERTA  ·  PROACT OBJECTIVES",
                units="How each objective is measured, and how it enters the optimization (Alberta portion of the Y2Y region)")
dp.values_table_simple(C, OUT / "01b_values_table_simple.png")     # the bare-bones version for the slides


In [ ]:
# ---- 2. Act 1 — the core at 1 km, two maps: (a) F over unprotected land with the Upper Smoky areas, (b) the regional core clusters
#         (numbered north -> south); the Alberta frame at left + ONE zoom inset on the key cluster at right (dp.STYLE["inset_clusters"]) ----
if KEY is not None:
    dp.core_map_F(C, OUT / "02a_act1_core_F.png")
    dp.core_map_clusters(C, OUT / "02b_act1_core_clusters.png")
else:
    print("no core clusters at the applied band: the core is (nearly) empty on the curated block -- no Act 1 maps; see 13's summary tiers map")


In [ ]:
# ---- 3. Act 1 — star plots of the core clusters + one locator window per cluster (aligned under the stars; each also as its own file) ----
if len(C.numbered("Act 1")):
    dp.core_stars(C, OUT / "03_act1_core_stars.png")
    dp.cluster_locators(C, OUT / "03b_act1_cluster_locators.png", panel_px=620)
else:
    print("no core clusters -- no stars / locators")


In [ ]:
# ---- 4. Act 1 — consequences: mean value in the cluster / mean over Alberta's allocatable land ("2.3x"); reference columns = existing PAs,
#         the Upper Smoky Nature-First zone and planning area (unprotected parts) ----------------------------------------------------------
if len(C.numbered("Act 1")):
    dp.core_consequences(C, OUT / "04_act1_consequences.png")
else:
    print("no core clusters -- no consequences table")


## Next outputs to add (say the word)
Act 2 pairings (value | tier) per scenario · Act 2 stars and consequences · the hinge cross-tab · the Act 3 gap map · the summary tiers map.